# Finding Points of Interest

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/03-points-of-interest.ipynb)

This notebook teaches you how to discover and analyze points of interest (POIs):

- POI categories and filters
- Distance-based queries
- Travel-time bounded search
- Analyzing POI data
- Practical applications

## Setup

In [ ]:
!pip install -q socialmapper[routing]

In [ ]:
import os
os.environ["SOCIALMAPPER_DEMO_MODE"] = "true"

from socialmapper import get_poi, create_isochrone
print("Ready!")

## What is a Point of Interest?

A POI is any location that might be useful or interesting:
- Restaurants, cafes, bars
- Hospitals, pharmacies, clinics
- Schools, libraries, universities
- Grocery stores, supermarkets
- Parks, gyms, community centers

SocialMapper queries OpenStreetMap for POI data.

## Basic POI Query

In [ ]:
# Find POIs near a location
pois = get_poi(
    location="Seattle, WA",
    limit=20
)

print(f"Found {len(pois)} points of interest:")
for poi in pois[:5]:
    print(f"  {poi['name']}: {poi['category']} ({poi['distance_km']:.2f} km)")

## POI Result Structure

Each POI contains detailed information.

In [ ]:
# Get a single POI and examine its structure
pois = get_poi("Portland, OR", categories=["cafe"], limit=1)
poi = pois[0]

print("POI structure:")
print(f"  name: {poi['name']}")
print(f"  category: {poi['category']}")
print(f"  lat: {poi['lat']}")
print(f"  lon: {poi['lon']}")
print(f"  distance_km: {poi['distance_km']:.3f}")
print(f"  address: {poi.get('address', 'N/A')}")
print(f"  tags: {poi['tags']}")

## Filtering by Category

Available categories:

| Group | Categories |
|-------|------------|
| **Food** | `restaurant`, `cafe`, `fast_food`, `bar`, `pub` |
| **Shopping** | `grocery`, `supermarket`, `convenience`, `mall` |
| **Healthcare** | `hospital`, `clinic`, `pharmacy`, `doctors`, `dentist` |
| **Education** | `school`, `university`, `college`, `library` |
| **Finance** | `bank`, `atm` |
| **Recreation** | `park`, `playground`, `gym`, `sports_centre` |

In [ ]:
# Find specific categories
location = "Chicago, IL"

# Single category
hospitals = get_poi(location, categories=["hospital"], limit=10)
print(f"Hospitals: {len(hospitals)}")

# Multiple categories
food_places = get_poi(
    location,
    categories=["restaurant", "cafe", "fast_food"],
    limit=30
)
print(f"Food places: {len(food_places)}")

# Show breakdown
from collections import Counter
categories = Counter(p['category'] for p in food_places)
for cat, count in categories.most_common():
    print(f"  {cat}: {count}")

## Healthcare Access Analysis

In [ ]:
location = "Atlanta, GA"

print(f"Healthcare Access in {location}:")
print("=" * 40)

# Hospitals
hospitals = get_poi(location, categories=["hospital"], limit=5)
print("\nNearest Hospitals:")
for h in hospitals:
    print(f"  {h['name']}: {h['distance_km']:.2f} km")

# Pharmacies
pharmacies = get_poi(location, categories=["pharmacy"], limit=10)
print(f"\nNearest Pharmacies ({len(pharmacies)} found):")
for p in pharmacies[:5]:
    print(f"  {p['name']}: {p['distance_km']:.2f} km")

# Clinics
clinics = get_poi(location, categories=["clinic"], limit=10)
print(f"\nClinics: {len(clinics)} found")

## Travel-Time Bounded Search

Find POIs within a travel-time boundary (uses isochrones internally).

In [ ]:
# Find restaurants within 15-minute walk
walkable_restaurants = get_poi(
    location="San Francisco, CA",
    categories=["restaurant"],
    travel_time=15,  # minutes
    limit=50
)

print(f"Restaurants within 15-min walk: {len(walkable_restaurants)}")

# Show distance distribution
distances = [r['distance_km'] for r in walkable_restaurants]
print(f"\nDistance range: {min(distances):.2f} - {max(distances):.2f} km")
print(f"Average distance: {sum(distances)/len(distances):.2f} km")

## Using Coordinates

Query POIs near specific coordinates.

In [ ]:
# Central Park, NYC coordinates
central_park = (40.7829, -73.9654)

# Find cafes nearby
cafes = get_poi(
    location=central_park,
    categories=["cafe"],
    limit=15
)

print(f"Cafes near Central Park: {len(cafes)}")
for cafe in cafes[:5]:
    print(f"  {cafe['name']}: {cafe['distance_km']:.2f} km")

## Extracting Additional Information

POI tags contain extra details from OpenStreetMap.

In [ ]:
# Get restaurants with details
restaurants = get_poi(
    "Austin, TX",
    categories=["restaurant"],
    limit=20
)

print("Restaurant Details:")
print("=" * 60)

for r in restaurants[:5]:
    name = r['name']
    distance = r['distance_km']
    cuisine = r['tags'].get('cuisine', 'Unknown cuisine')
    website = r['tags'].get('website', 'No website')
    phone = r['tags'].get('phone', 'No phone')
    
    print(f"\n{name}")
    print(f"  Distance: {distance:.2f} km")
    print(f"  Cuisine: {cuisine}")
    print(f"  Website: {website[:50]}..." if len(website) > 50 else f"  Website: {website}")

## School Proximity Analysis

In [ ]:
# Analyze school access for a residential area
home = (41.8781, -87.6298)  # Chicago

schools = get_poi(
    location=home,
    categories=["school"],
    limit=30
)

# Categorize by distance
walking = [s for s in schools if s['distance_km'] <= 1.0]
short_drive = [s for s in schools if 1.0 < s['distance_km'] <= 3.0]
farther = [s for s in schools if s['distance_km'] > 3.0]

print("School Accessibility Analysis:")
print("=" * 40)
print(f"Within 1 km (walking): {len(walking)}")
for s in walking:
    print(f"  - {s['name']}: {s['distance_km']:.2f} km")

print(f"\nWithin 3 km (short drive): {len(short_drive)}")
print(f"Beyond 3 km: {len(farther)}")

## Food Desert Identification

In [ ]:
def check_food_access(location, travel_time=15):
    """Check if an area has adequate grocery access."""
    
    groceries = get_poi(
        location,
        categories=["grocery", "supermarket"],
        travel_time=travel_time,
        limit=50
    )
    
    print(f"Food Access Analysis: {location}")
    print("=" * 40)
    print(f"Grocery stores within {travel_time}-min walk: {len(groceries)}")
    
    if groceries:
        distances = [g['distance_km'] for g in groceries]
        print(f"Nearest store: {min(distances):.2f} km")
    
    # Assessment
    if len(groceries) < 2:
        status = "POTENTIAL FOOD DESERT"
    elif len(groceries) < 5:
        status = "LIMITED ACCESS"
    else:
        status = "GOOD ACCESS"
    
    print(f"\nAssessment: {status}")
    return groceries

# Test different areas
check_food_access("Detroit, MI")
print()
check_food_access("Manhattan, NY")

## Business Competition Analysis

In [ ]:
# Evaluate competition for a potential coffee shop location
potential_location = (47.6062, -122.3321)  # Seattle downtown

competitors = get_poi(
    potential_location,
    categories=["cafe"],
    limit=50
)

# Analyze competition density
within_500m = [c for c in competitors if c['distance_km'] <= 0.5]
within_1km = [c for c in competitors if c['distance_km'] <= 1.0]

print("Competition Analysis for New Coffee Shop:")
print("=" * 45)
print(f"Competitors within 500m: {len(within_500m)}")
print(f"Competitors within 1km: {len(within_1km)}")

# Show nearby competitors
if within_500m:
    print("\nNearest competitors:")
    for c in sorted(within_500m, key=lambda x: x['distance_km'])[:5]:
        print(f"  {c['name']}: {c['distance_km']*1000:.0f}m")

# Recommendation
if len(within_500m) > 5:
    print("\nRecommendation: HIGH competition - consider another location")
elif len(within_500m) < 2:
    print("\nRecommendation: LOW competition - good opportunity!")
else:
    print("\nRecommendation: MODERATE competition - viable location")

## Combining POIs with Isochrones

In [ ]:
from shapely.geometry import shape, Point

# Create isochrone
location = "Minneapolis, MN"
isochrone = create_isochrone(location, travel_time=10, travel_mode="walk")
polygon = shape(isochrone['geometry'])

# Get restaurants in a larger area
restaurants = get_poi(location, categories=["restaurant"], limit=100)

# Filter to only those inside the isochrone
accessible = []
for r in restaurants:
    point = Point(r['lon'], r['lat'])
    if polygon.contains(point):
        accessible.append(r)

print(f"Restaurants within 10-min walk: {len(accessible)}")
print(f"Total restaurants found: {len(restaurants)}")
print(f"Accessibility rate: {len(accessible)/len(restaurants)*100:.1f}%")

## Amenity Summary Report

In [ ]:
def generate_amenity_report(location):
    """Generate a comprehensive amenity report for a location."""
    
    categories = {
        "Food": ["restaurant", "cafe", "grocery"],
        "Healthcare": ["hospital", "pharmacy", "clinic"],
        "Education": ["school", "library"],
        "Recreation": ["park", "gym"]
    }
    
    print(f"\nAmenity Report: {location}")
    print("=" * 50)
    
    for group, cats in categories.items():
        pois = get_poi(location, categories=cats, limit=50)
        
        if pois:
            nearest = min(p['distance_km'] for p in pois)
            print(f"\n{group}:")
            print(f"  Total found: {len(pois)}")
            print(f"  Nearest: {nearest:.2f} km")
            
            # Show top 3
            for p in sorted(pois, key=lambda x: x['distance_km'])[:3]:
                print(f"    - {p['name']}: {p['distance_km']:.2f} km")
        else:
            print(f"\n{group}: None found nearby")

# Generate report
generate_amenity_report("Denver, CO")

## Exercise: Find Your Ideal Neighborhood

In [ ]:
def score_neighborhood(location):
    """Score a neighborhood based on walkable amenities."""
    
    score = 0
    details = []
    
    # Groceries (most important)
    groceries = get_poi(location, categories=["grocery", "supermarket"], travel_time=10, limit=10)
    if len(groceries) >= 2:
        score += 30
        details.append(f"Groceries: {len(groceries)} stores (+30)")
    elif len(groceries) == 1:
        score += 15
        details.append(f"Groceries: 1 store (+15)")
    else:
        details.append("Groceries: None (0)")
    
    # Restaurants
    restaurants = get_poi(location, categories=["restaurant", "cafe"], travel_time=10, limit=30)
    if len(restaurants) >= 10:
        score += 20
        details.append(f"Dining: {len(restaurants)} places (+20)")
    elif len(restaurants) >= 5:
        score += 10
        details.append(f"Dining: {len(restaurants)} places (+10)")
    
    # Parks
    parks = get_poi(location, categories=["park"], travel_time=10, limit=10)
    if parks:
        score += 20
        details.append(f"Parks: {len(parks)} nearby (+20)")
    
    # Healthcare
    healthcare = get_poi(location, categories=["pharmacy"], travel_time=15, limit=5)
    if healthcare:
        score += 15
        details.append(f"Pharmacy: {len(healthcare)} nearby (+15)")
    
    # Transit
    transit = get_poi(location, categories=["bus_station", "train_station"], travel_time=10, limit=10)
    if transit:
        score += 15
        details.append(f"Transit: {len(transit)} stops (+15)")
    
    return score, details

# Compare neighborhoods
neighborhoods = [
    "Capitol Hill, Seattle",
    "Ballard, Seattle",
    "Fremont, Seattle"
]

print("Neighborhood Walkability Scores:")
print("=" * 50)

results = []
for n in neighborhoods:
    score, details = score_neighborhood(n)
    results.append((n, score, details))
    print(f"\n{n}: {score}/100")
    for d in details:
        print(f"  {d}")

# Winner
winner = max(results, key=lambda x: x[1])
print(f"\nMost walkable: {winner[0]} ({winner[1]}/100)")

## Next Steps

Continue with:

- **[Census Data](04-census-data.ipynb)** - Add demographic context
- **[Mapping](05-mapping-visualization.ipynb)** - Visualize POI locations
- **[Complete Workflow](06-complete-workflow.ipynb)** - Full analysis examples